# 02 — Does test look like train?

Workflow step 3. Everything after this depends on one premise:

> a held-out fold of training rows is a fair stand-in for the test set

If that is false, the CV score measures performance on a population we are not scored on,
and six weeks of tuning chase a number that does not track the leaderboard. This notebook
tests the premise three ways, from weakest evidence to strongest.

| | Question | Can see |
|---|---|---|
| 1 | Does any **column** differ? | marginals only |
| 2 | Do nulls **clump on rows**? | one specific joint property |
| 3 | Does **anything** differ? | the full joint distribution |

One quantity is used throughout. $N_{tr}$ and $N_{te}$ are the row counts, and

$$\pi \;=\; \frac{N_{te}}{N_{tr} + N_{te}} \;=\; \frac{295{,}753}{985{,}841} \;=\; 0.300$$

is the share of all rows that came from test. Under no shift, **every** subset of rows —
every bin, every category, every leaf of a tree — should be about $\pi$ test. Most of what
follows is one question asked of different subsets: *is this group 30% test?*

All logic lives in `src/s6e7/`; this notebook only calls and displays.

In [17]:
%load_ext autoreload
%autoreload 2

import polars as pl

from s6e7 import adversarial, eda, io

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_float_precision(6)

train, test = io.load_train(), io.load_test()

# None = full data (the real answer). Set to 0.1 for a ~30 s pass while iterating;
# a subsample can only ever WEAKEN evidence of a shift, never manufacture it.
SAMPLE = None

print(f"train {train.height:,} rows   test {test.height:,} rows")
print(f"pi (global test share) = {test.height / (train.height + test.height):.6f}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
train 690,088 rows   test 295,753 rows
pi (global test share) = 0.300001


---
## 0. Why `id` is excluded from everything below

Competition ids are handed out per file, so they occupy disjoint contiguous ranges. A
single split on `id` separates train from test perfectly — AUC 1.0 that says nothing
whatever about the features. Confirm it once, then never feed it to the classifier.

In [18]:
a, b = train["id"], test["id"]
print(f"train id  {a.min():>7,} – {a.max():>7,}   ({a.n_unique():,} unique)")
print(f"test  id  {b.min():>7,} – {b.max():>7,}   ({b.n_unique():,} unique)")
print(f"overlap   {len(set(a.to_list()) & set(b.to_list()))}")

train id        0 – 690,087   (690,088 unique)
test  id  690,088 – 985,840   (295,753 unique)
overlap   0


---
## 1. Do any columns differ? — the marginal check

The check most people mean by "plot train vs test". Five numbers per column, all from
`eda.numeric_shift`. Each exists to catch a failure the others are blind to.

**`null_gap` — do both files lose this column equally often?**

```python
null_gap = 100 * test[col].null_count() / test.height \
         - 100 * train[col].null_count() / train.height
```

`bmi`: train 13,898 / 690,088 = 2.01395%, test 5,956 / 295,753 = 2.01384%, gap
**−0.00010 pp**. Two files of very different size agreeing to five decimals is not "two
similar datasets" — it is one generator run twice with the same null probability. All 13
columns look like this, which is exactly what makes section 2 so strange.

**`gap_sd` — are the two centres in the same place?**

```python
gap_sd = (test[col].mean() - train[col].mean()) / train[col].std()
```

Dividing by the column's own spread is the whole point: it turns "0.6 steps" and "0.01
hours" into one comparable unit — *typical deviations apart*. Largest here: 0.0057.

**`sd_ratio` — are they equally wide?** `test[col].std() / train[col].std()`, where 1.0 is
identical. It exists because two distributions can share a mean and differ completely in
width, a case `gap_sd` scores as exactly zero. All seven land within 0.8% of 1.0.

**`max_bin_dev` — does any slice of the range hold too many test rows?**

The recipe:

1. Pool both files' values for the column and sort them.
2. Cut into 50 buckets holding equal numbers of rows — lowest 2%, next 2%, …
3. In each bucket, what fraction of the rows came from test?
4. Every bucket should sit at π = 0.300. Report the worst.

**Why bucket at all, when we already have the mean and the spread?** Because both collapse
the column to a single number, and a distribution can move a lot without moving either.
Picture a 0–100 column where test has a hole between 40 and 50, those rows spread evenly
over the rest of the range. The mean is unchanged. The spread barely moves. But the five
buckets covering 40–50 contain no test rows at all, and `max_bin_dev` reads 0.30.

Bucketing keeps **50 numbers instead of 1**, so it can say *where in the range* the
imbalance sits. That is the only one of the five that sees a change in **shape**.

`heart_rate`, the worst of the seven:

    worst bucket = lowest 2% of heart rates, 50.0 – 58.6 bpm
    19,421 rows:  13,900 train  +  5,521 test
    expected test 5,826   ->   actual 5,521   ->   305 short
    share 0.2843  vs  0.3000   ->   max_bin_dev = 0.0157

Centre and width match train's exactly, and the bottom bucket is *still* short.

**`p_value` — could that bucket just have been unlucky?**

Same 50 buckets. Each has an observed test count and an expected one (bucket size × 0.300).
Square the difference and divide by the expected count — dividing is what makes buckets
comparable, since being 300 rows short of an expected 5,900 is routine while being 300
short of an expected 400 is not. Sum across all 50 buckets. That total is the **chi-square
statistic**.

Then the step that turns it into a probability: **we know what that total looks like when
nothing is wrong.** With 50 buckets, pure randomness produces a total of about 49 on
average, and rarely more than about 66. So

> p-value = *of all the totals randomness alone could have produced, what fraction are at
> least as large as the one I got?*

`sleep_duration` gives 0.9405 — nine times out of ten, chance does worse. `heart_rate`
gives 0.0002 — 2 chances in 10,000.

In [19]:
eda.numeric_shift(train, test, io.NUMERIC_COLS).select(
    "column", "gap_sd", "sd_ratio", "null_gap", "p_value", "max_bin_dev"
)

column,gap_sd,sd_ratio,null_gap,p_value,max_bin_dev
str,f64,f64,f64,f64,f64
"""heart_rate""",-0.002073,0.992662,-0.000004,0.000189,0.015721
"""step_count""",0.002717,0.998790,-0.000007,0.009961,0.009218
"""exercise_duration""",0.003070,0.997897,0.000142,0.125036,0.008700
"""water_intake""",0.005720,1.000543,-0.000021,0.795945,0.008111
"""calorie_expenditure""",-0.001640,1.000819,0.000216,0.654987,0.007438
"""sleep_duration""",0.000757,1.000792,-0.000037,0.940500,0.006439
"""bmi""",-0.000106,0.998760,-0.000103,0.605459,0.005859


### Is `p < 0.05` enough to call it real? No — and here is why

**We ran seven tests, not one.** If all seven columns were perfectly clean, the chance that
*at least one* trips a 0.05 wire is $1 - 0.95^{7} = 30\%$. Finding one small p among seven
is what you should expect. The crude correction is to divide the threshold by the number of
tests: $0.05 / 7 = 0.007$.

    heart_rate   p = 0.0002   clears 0.007  ->  real
    step_count   p = 0.0099   does not      ->  back in the "unexplained" pile

**And 0.05 was never a fact about the world.** It is a convention about how much
false-positive risk to accept. Nothing about this dataset makes 0.05 the right number.

### What does a *large* p-value license? Not "nothing is wrong"

This is the trap. A large p-value normally means only *we failed to detect anything* — the
test may simply be too weak to see what is there. So "p = 0.94, therefore no shift" is an
unearned conclusion **unless you know what the test could have caught.**

We can measure that instead of assuming it. `eda.shift_power` injects a shift of known size
and counts how often the same chi-square fires:

In [ ]:
one_bucket = eda.shift_power(985_841, shifted_bins=1)      # hardest: one bucket drifts
whole_col  = eda.shift_power(                              # easiest: the whole column drifts
    985_841, shifted_bins=None, deltas=(0.0, 0.001, 0.002, 0.003, 0.005)
)
pl.concat([whole_col, one_bucket])

Reading the two blocks:

- A **whole-column drift** of 0.3 pp is caught 96% of the time. Anything broader than that
  cannot hide from this test.
- A **single-bucket spike** needs about 2.0 pp before it is caught 92% of the time. A 1.0 pp
  spike is *missed* three times in four — 49 well-behaved buckets dilute one bad one.

So `sleep_duration`'s p = 0.9405 licenses a real, bounded conclusion — **there is no broad
drift above roughly 0.3 pp in this column** — and does *not* license "nothing is wrong
anywhere", because a narrow spike could still be hiding under it. **Power is what converts
"we found nothing" into a bound.** Without it, a large p-value means very little.

### So how do we say a `max_bin_dev` is *big*?

**Against the noise floor, not against zero** — and this is the part I had wrong when the
table first appeared above.

Look at the `delta = 0.0` rows: with no shift whatsoever, the median `max_bin_dev` is
**0.0081**. With 50 buckets, the luckiest one always drifts a little. *That* is the
reference, not zero:

    bmi                   0.0059    BELOW the floor — quieter than chance usually manages
    sleep_duration        0.0064    at the floor
    calorie_expenditure   0.0074    at the floor
    water_intake          0.0081    exactly the floor
    exercise_duration     0.0087    at the floor
    step_count            0.0092    marginal
    heart_rate            0.0157    ~2x the floor — the only column that clears it

Six of the seven are indistinguishable from no shift at all.

Two further rungs, once a column *does* clear the floor:

1. **Relative size.** $\delta / \pi$. `heart_rate`'s 0.0157 against a base of 0.300 means
   that slice of the range is about **5% over- or under-represented** in test. Below
   roughly 10% relative is rarely worth acting on.
2. **Does it change a decision?** Not answerable one column at a time — and it is exactly
   what section 3 answers directly. **`max_bin_dev` localises; the adversarial AUC decides.**
   Read the AUC first: if it is 0.5, nothing in this table matters.

For categoricals there is nothing to bin — **the levels already are the bins.** `diff` is
each level's share of its own file in percentage points, with nulls counted as their own
level so every column's shares sum to 100.

In [20]:
eda.category_shift(train, test, io.CATEGORICAL_COLS).head(10)

column,level,train_pct,test_pct,diff,abs_diff
str,str,f64,f64,f64,f64
"""gender""","""female""",32.461947,29.152705,-3.309242,3.309242
"""gender""","""other""",29.987915,33.114795,3.126881,3.126881
"""physical_activity_level""","""active""",30.813751,29.957431,-0.856321,0.856321
"""physical_activity_level""","""sedentary""",31.848692,32.449375,0.600683,0.600683
"""physical_activity_level""","""moderate""",32.030842,32.286401,0.255559,0.255559
"""smoking_alcohol""","""yes""",32.420503,32.216410,-0.204093,0.204093
"""gender""","""male""",34.452997,34.635321,0.182323,0.182323
"""sleep_quality""","""average""",31.003003,30.870017,-0.132986,0.132986
"""smoking_alcohol""","""no""",31.849706,31.978712,0.129006,0.129006


**Verdict on the marginals: they pass.** Mean gaps under 0.006 SD, spread ratios within
0.8%, null rates identical to five decimals. The only visible movement is `gender`
(female −3.3 pp, other +3.1 pp) and `physical_activity_level` (under 1 pp).

If we stopped here we would write "train and test are one distribution" and freeze random
folds. Section 3 shows that would be wrong.

---
## 2. Do nulls clump on the same rows? — one joint property

Everything in section 1 was about **columns**. This is about **rows**. For each row, count
how many of its 13 fields are missing; call that count $K$.

Section 1 already pinned down the *average* of $K$ — both files lose each column at the
same rate, so both must average the same nulls per row. What it left completely open is
how those nulls are **arranged**. The same total can be sprinkled one-per-row across many
rows, or piled several-per-row onto few.

If each column drops values independently of the others, we can compute exactly what $K$
should look like. No simulation:

```python
pmf = np.array([1.0])          # before any column: "0 nulls", with certainty
for p in rates:                # one column at a time
    pmf = np.convolve(pmf, [1.0 - p, p])
```

Read it as bookkeeping. Start knowing a row has 0 nulls. Bring in a column: every
possibility you are holding splits in two — the column is *present* (weight $1-p$, count
unchanged) or *missing* (weight $p$, count $+1$). Thirteen columns, thirteen steps, and
you have the exact probability of every value of $K$ — rather than enumerating all
$2^{13} = 8{,}192$ null masks. That is the `independent_pct` column, built from **train's
own** rates.

Then the part that does the work:

$$\mathbb{E}[K] = \sum_j p_j \qquad\qquad \operatorname{Var}(K) = \sum_j p_j\,(1 - p_j)$$

**The mean is blind to clumping. Only the variance sees it.** Move nulls off one row and
onto another: the column rates never change, so $\mathbb{E}[K]$ never changes — but $K$
spreads out. So the entire diagnostic is one ratio, observed variance over
$\sum_j p_j(1-p_j)$: about 1 means nulls fall independently, above 1 means they clump.

In [ ]:
eda.missingness_dispersion(train, test, io.FEATURE_COLS)

In [ ]:
eda.null_count_profile(train, test, io.FEATURE_COLS)

**There it is.** Same mean number of nulls per row to six decimals (0.651360 vs 0.651361),
but test's **variance is 32% above** what independence predicts, while train sits right on
it (0.9925×).

Test's missingness is **clustered**: more perfectly-complete rows *and* more
heavily-gutted rows, fewer rows with exactly one gap. Identical columns, different rows —
which is precisely the thing no per-column check could ever have reported.

---
## 3. Does anything at all differ? — adversarial validation

Sections 1 and 2 each tested something specific, chosen in advance. This tests
*everything at once*, including whatever we did not think to look for.

Throw the real target away. Label each row 0 if it came from `train.csv` and 1 if from
`test.csv`, then fit $\hat q(x) \approx P(y = 1 \mid x)$ out-of-fold — exactly as we would
fit any model. Score it with AUC, which reads directly as a probability:

$$\text{AUC} \;=\; P\bigl(\hat q(x^{te}) > \hat q(x^{tr})\bigr)$$

*Draw one test row and one train row at random; how often does the model rank them the
right way round?* At 0.5 it cannot order them at all — **nothing** distinguishes the files,
random folds are fair, proceed. Higher means a difference exists somewhere in the joint
distribution.

Because AUC only ever compares one test row against one train row, the 70/30 imbalance
cancels out and needs no correction.

This turns "are these two 986,000-row, 13-dimensional distributions the same?" into
"what's the AUC?" — a question we already have tools for.

In [14]:
result = adversarial.run(train, test, sample_frac=SAMPLE)
print(result.report())

adversarial validation: 690,088 train vs 295,753 test, 13 features
  OOF AUC   0.6518   (per-fold sd 0.0012)
  per fold  0.6498, 0.6530, 0.6519, 0.6512, 0.6532

SHIFT — AUC 0.6518 > 0.55. Read the importances; revisit fold design before freezing.

  feature                     gain   splits
  water_intake              32.34%   10,677
  calorie_expenditure       19.90%    9,713
  bmi                       12.78%   13,119
  physical_activity_level   10.86%    4,187
  smoking_alcohol            7.29%    3,586
  step_count                 5.23%   12,063
  diet_type                  3.24%    2,248
  gender                     2.61%    2,441
  exercise_duration          1.54%   10,026
  sleep_duration             1.51%   10,770
  heart_rate                 1.42%    9,946
  sleep_quality              1.06%    2,535
  stress_level               0.22%    1,689


### Do NOT read that importance table as a shift ranking

Gain totals the loss reduction across the splits a feature was chosen for, so it counts
**opportunities** as much as signal. A continuous column with 12,000 distinct values
offers vastly more candidate thresholds than a 3-level categorical, and gain rewards that
whatever the truth is.

The honest per-feature number is that feature's adversarial AUC **on its own** — same
definition as above, same scale as the joint result, one column in the model:

In [15]:
solo = adversarial.solo_auc(train, test, sample_frac=SAMPLE)
solo.join(result.importance.select("feature", "gain_pct"), on="feature").sort(
    "solo_auc", descending=True
)

feature,solo_auc,gain_pct
str,f64,f64
"""gender""",0.521587,2.614562
"""physical_activity_level""",0.504821,10.856737
"""heart_rate""",0.501033,1.415346
"""step_count""",0.500797,5.228176
"""smoking_alcohol""",0.500453,7.287704
"""sleep_quality""",0.500327,1.060714
"""stress_level""",0.500156,0.222917
"""diet_type""",0.500014,3.236705
"""calorie_expenditure""",0.499625,19.902258


Every feature alone is at chance; together they are not — best solo **0.522** against a
joint **0.652**. **That gap is the part of the shift no single column carries**: the part
only a joint search can find, and the part no plot can show.

Note how badly gain ranks it. `water_intake` takes 32% of the gain and is at chance on its
own; `gender` takes 2.6% and is the most-shifted column in the table. Here the gain
ranking is close to *anti-correlated* with the truth.

---
## 4. The control — is the harness lying?

A positive adversarial result is a claim about the data. This is what stops it being a
claim about our code. Permute the labels — destroying any real association while keeping
the class balance exactly — then re-run the whole pipeline:

```python
y_shuffled = np.random.default_rng(seed).permutation(y)
```

The shuffled label is independent of the features by construction, so the only achievable
score is 0.5. Any bug that leaks the label — a misaligned concat, a row-order assumption,
an index mismatch — survives permutation and shows up here as an AUC above chance.

**Must come back ≈ 0.5000.** Run it every time the headline AUC is not already 0.5.

In [16]:
control = adversarial.shuffled_control(train, test, sample_frac=SAMPLE)
print(f"shuffled-label AUC  {control:.4f}   (must be ~0.5000)")

shuffled-label AUC  0.5002   (must be ~0.5000)


---
## Conclusion

| Check | Result |
|---|---|
| Numeric marginals | pass — centres within 0.006 SD |
| Null rates per column | pass — identical to 5 decimals |
| Category levels | near-pass — `gender` moves 3.3 pp |
| Correlations | pass — largest difference 0.006 |
| Solo adversarial AUC | pass — every feature at chance |
| **Nulls per row** | **fail — variance 1.32× independence** |
| **Joint adversarial AUC** | **fail — 0.6518, control 0.5002** |

Train and test are **not** draws from one distribution. Train draws nulls independently
per column; test lets them co-occur.

**Folds are not frozen.** The i.i.d. premise behind a plain stratified split is dead as
stated, and the fold design has to answer this shift before step 4.
